~~1. 修改可视化代码~~
~~2. 从A-->V开始归因~~
~~3. 编写K归因代码~~
4. 适配transformerlens代码

In [ ]:
# 自动重载本地模块，便于边改边跑
%load_ext autoreload
%autoreload 2
%aimport -graph_registry  # Exclude from autoreload!

from IPython.core.interactiveshell import InteractiveShell
# 让 Jupyter 单元格显示所有表达式结果
InteractiveShell.ast_node_interactivity = 'all'
from ipykernel import get_connection_file
# 打印当前 kernel 连接信息，方便排查多 kernel 场景
print(get_connection_file())

In [ ]:
import sys
import os
os.environ["CUDA_DEVICE_ORDER"]="PCI_BUS_ID"   
os.environ["CUDA_VISIBLE_DEVICES"]="0"
import json
from functools import partial
import itertools
from itertools import product, chain
from typing import Tuple, List, Optional
from collections import Counter
from dataclasses import dataclass
import math

import circuitsvis as cv
from circuitsvis.tokens import colored_tokens, colored_tokens_multi

from pptree import Node as TNode, print_tree
from common_utils import join_lists, topk_md, numpy, show_topk, mr

from llm import *
from min_arc import *
from model_hooks import *
from attribute import *
from vis import *
from seeds.common import *
# from seeds.test import *

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F 
import einops
from einops import rearrange
import transformers
from transformers import AutoConfig, AutoModelForCausalLM, AutoTokenizer, GPTQConfig, BitsAndBytesConfig
_ = torch.set_grad_enabled(False)

In [ ]:
# 统一缓存已加载模型，避免重复加载
models = {}

In [ ]:
# bitsandbytes 4bit 量化配置（NF4 + fp16 计算）
bnbconfig = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type='nf4'
)

In [ ]:
# 加载 HF 基座模型（后续可用于与 TL 模型结果对齐）
device = 'cuda:0'
model_name = "Qwen/Qwen3-32B"
model_base = AutoModelForCausalLM.from_pretrained(model_name,
    # device_map="cpu",
    attn_implementation="eager",
    output_hidden_states=True,
    device_map='auto',
    quantization_config=bnbconfig,
    low_cpu_mem_usage=True,
    torch_dtype=torch.float16,
    local_files_only=True,
    trust_remote_code=True,
)
# tokenizer 绑定到模型对象，便于后续统一调用
tokenizer = AutoTokenizer.from_pretrained(model_name)
model_base.tokenizer = tokenizer

In [ ]:
models[model_name] = model_base

space_token = 'Ġ';  newline_token = 'Ċ'  # for BPE tokenizers
L, H, V = model_base.config.num_hidden_layers, model_base.config.num_attention_heads, model_base.config.vocab_size

In [ ]:
import transformer_lens
print(transformer_lens.__file__)
from transformer_lens import HookedTransformer
import transformer_lens.utils as utils
from transformer_lens.utils import repeat_along_head_dimension
import gc

In [ ]:
# 将同一 HF 权重加载成 TransformerLens 结构，方便做可解释性分析
model = HookedTransformer.from_pretrained(model_name,
    hf_model=model_base,
    use_split_qkv_attention=False,
    load_in_4bit=True,
    tokenizer=tokenizer,
    device='cuda',
    device_map='auto',
    # n_devices=2,
    move_to_device=True,
    torch_dtype=torch.float16,
    center_writing_weights=False,
    center_unembed=False,
    fold_ln=False,
    fold_value_biases=False,
    trust_remote_code=True,
)
# 记录到模型缓存字典
models[model_name] = model

# 使用 TL 配置重新刷新模型维度信息
space_token = 'Ġ';  newline_token = 'Ċ'  # for BPE tokenizers
L, H, V = model.cfg.n_layers, model.cfg.n_heads, model.cfg.d_vocab

In [ ]:
# 释放不再使用的 HF 基座模型显存

del model_base
gc.collect()
torch.cuda.empty_cache()

In [ ]:
# （可选分支）加载另一套 GPTQ 模型用于对比实验

device = 'cuda:0'
cache_dir = '/data0/modelscope/'
# model_name = 'Qwen/Qwen2.5-72B-Instruct'
# model_name = 'Qwen/Qwen2.5-72B-Instruct-AWQ'
model_name = 'Qwen/Qwen2.5-72B-Instruct-GPTQ-Int4'
# model_name = 'Qwen/Qwen2.5-72B-Instruct'
if model_name not in models:
    model_dir = cache_dir + model_name
    model_dir = model_name
    gptq_config = GPTQConfig(bits=4, group_size=128, desc_act=False, use_exllama=True, exllama_config={"version": 2})
    model = AutoModelForCausalLM.from_pretrained(model_dir,
        # load_in_4bit=True, bnb_4bit_quant_type='nf4', #use_safetensors=False)
        torch_dtype=torch.float16, quantization_config=GPTQConfig, trust_remote_code=True,
        local_files_only=True, low_cpu_mem_usage=True, device_map=device)
    tokenizer = AutoTokenizer.from_pretrained(model_dir)
    model.tokenizer = tokenizer
    models[model_name] = model
    
# 刷新 tokenizer 与模型维度信息
tokenizer = AutoTokenizer.from_pretrained(model_dir)
space_token = 'Ġ';  newline_token = 'Ċ'  # for BPE tokenizers
L, H, V = model.config.num_hidden_layers, model.config.num_attention_heads, model.config.vocab_size

In [ ]:
import ast
# 要评估的模型标识（写入 Result.model 字段）
api_models = [
    'qwen3-32b',
]

# 读取 Hi-ToM 数据，并把 index 字段还原成 Python 结构
hi_tom_path = 'project/BARC-transformerlens/data_uniform/Hi_ToM_order_1.csv'
df_tom = pd.read_csv(hi_tom_path)
df_tom['index'] = df_tom['index'].apply(ast.literal_eval)

# 控制样本规模（至少 20 条）
n_samples = max(20, len(df_tom))
df_tom = df_tom.head(n_samples)

# 将每条数据整理成统一的 Result 对象
results = []
for i, row in df_tom.iterrows():
    prompt = row['prompt']
    answer = row['answer']
    choices_str = row['choices']
    index_map = [row['index']]
    choices = [c.split('. ')[1].strip() for c in choices_str.split(', ')]
    candidate_ids = [tokenizer.encode(' ' + choice)[0] for choice in choices]
    label = tokenizer.encode(' ' + answer)[0]
    
    prompt_tokens = tokenizer.encode(prompt)
    answer_idx = len(prompt_tokens)
    for api_model in api_models:
        results.append(Result(
            index=i,
            model=api_model,
            prompt=prompt,
            answers=[answer],
            answer_indices=[answer_idx],
            candidate_ids=candidate_ids,
            labels=[label],
            n_tokens=answer_idx,
            index_map=index_map,
        ))

In [ ]:
# run_with_cache 仅保存后续归因所需的关键 hook
names_filter = [
    f"blocks.{layer}.ln1.hook_scale" for layer in range(model.cfg.n_layers)
] + [
    f"blocks.{layer}.hook_mlp_out" for layer in range(model.cfg.n_layers)
] + [
    f"blocks.{layer}.hook_resid_pre" for layer in range(model.cfg.n_layers)
] + [
    f"blocks.{layer}.ln2.hook_scale" for layer in range(model.cfg.n_layers)
] + [
    f"ln_final.hook_scale"
] + [
    f"blocks.{layer}.attn.hook_z" for layer in range(model.cfg.n_layers)
] + [
    f"ln_final.hook_normalized"
]

In [ ]:
# 当前实验默认使用全部样本
_results = results

In [ ]:
# 用 TransformerLens 路径前向并缓存中间量，写回每条样本结果
for r in tqdm(results):
    # model_inputs = tokenizer([r.prompt], return_tensors="pt").to(model.cfg.device if model.cfg.device is not None else model.device)
    model_inputs = model.to_tokens(r.prompt, prepend_bos=False, padding_side='left')
    output_logits, cache = model.run_with_cache(model_inputs, names_filter=names_filter)
    r.outputs = get_outputs_from_cache(model, cache)
    # 记录预测文本、正确性与目标 token 对数概率
    r.responses, r.is_corrects, r.logprobs = [], [], []
    for answer, ans_idx, label in zip(r.answers, r.answer_indices, r.labels):
        logits = output_logits[0, ans_idx-1]
        response = tokenizer.decode(logits.argmax(dim=-1).item())
        r.responses.append(response)
        r.is_corrects.append(response.strip().strip('"').strip("'") == answer)
        r.logprobs.append(logits.log_softmax(dim=-1)[label].item())

In [ ]:
# 用 HF 原生路径前向，作为与 TL 路径对照

torch.cuda.empty_cache()
for r in tqdm(results):
    model_inputs = tokenizer([r.prompt], return_tensors="pt").to(model_base.device)
    set_hooks(model_base)
    output = model_base(**model_inputs, output_hidden_states=True)
    r.outputs = get_outputs(model_base, output)
    # 记录预测文本、正确性与目标 token 对数概率
    r.responses, r.is_corrects, r.logprobs = [], [], []
    for answer, ans_idx, label in zip(r.answers, r.answer_indices, r.labels):
        logits = output.logits[0, ans_idx-1]
        response = tokenizer.decode(logits.argmax(dim=-1).item())
        r.responses.append(response)
        r.is_corrects.append(response.strip().strip('"').strip("'") == answer)
        r.logprobs.append(logits.log_softmax(dim=-1)[label].item())

In [ ]:
# encoding = tokenizer(results[0].prompt)
# ids = encoding['input_ids']
# tokens = tokenizer.convert_ids_to_tokens(ids)

# # 显示对应关系
# for i, (token_id, token) in enumerate(zip(ids, tokens)):
#     print(f"{i:4d}: {token_id:6d} -> {token}")

In [ ]:
# 可在此按关系筛选样本；默认仍使用全部样本
_results = results # [r for r in results if r.rel_fn == 'right_of']; r = _results[-1]

In [ ]:
# 基于 HF 模型维度初始化归因图
graph = Graph(dataset_size=len(_results), hidden_size=model_base.config.hidden_size)

In [ ]:
# 基于 TL 模型维度初始化归因图，并补齐统一属性
graph = Graph(dataset_size=len(_results), hidden_size=model.cfg.d_model)
model.device = model.cfg.device
model.dtype = model.cfg.dtype

In [ ]:
import attribute
from contextlib import contextmanager

# 备份原逻辑：仅在需要时覆盖 dequant 上下文
_orig_use_dequant = attribute.use_dequant_projections

@contextmanager
def _patched_use_dequant_projections(module, layer, grad_inputs):
    # TL attention: 有 W_O，但没有 o_proj
    is_tl_attn = hasattr(module, "W_O") and not hasattr(module, "o_proj")
    if is_tl_attn:
        yield  # no-op，相当于不执行 with 内替换逻辑
        return
    with _orig_use_dequant(module, layer, grad_inputs):
        yield

# 将 patch 注入 attribute 模块
attribute.use_dequant_projections = _patched_use_dequant_projections

In [ ]:
# 构建根节点（lm_head）并初始化归因树
lmhead = Node(L, None, 'lm_head', attn_pattern='A-->A-')
nodes = [lmhead]
for n in nodes: graph.add_node(n)
# 优先使用 model_base；若不存在则回退到 model
selected_model = model_base if 'model_base' in globals() and model_base is not None else model
root = tnode = add_tnode(_results, selected_model, nodes); print_tree(root)

In [ ]:
# 对当前层归因结果做 head 聚类，并计算簇内/簇间指标
d = tnode.data
d.groups, d.metrics, _, _ = cluster_heads(d.attn_attrs_ds, threshold=0.5, strengths=d.top_heads, model_config=selected_model.config,
    figsize=(18, 5), width_ratios=(4, 1), bar_height=0.5, leaf_font_size=9)

In [ ]:
# 调整阈值后再次聚类，便于比较分组稳定性
d = tnode.data
d.groups, d.metrics, _, _ = cluster_heads(d.attn_attrs_ds, threshold=0.4, strengths=d.top_heads, model_config=selected_model.cfg,
    figsize=(18, 5), width_ratios=(4, 1), bar_height=0.5, leaf_font_size=9)

In [ ]:
# 随机抽样一个样本可视化指定 head 的注意力
colored_tokens_multi(*show_attn(random.choice(_results), selected_model, 51, 1, downstreams=tnode.data.nodes, start=_results[0].index_map[0]['start']))#, start=100))

In [ ]:
# 汇总每个 top head 的分数、准确率与多种 pattern 匹配得分
attn_patterns = ['A-->V', 'A-->A-', 'V->VK_C', 'V->VK_I', 'A-->QK_C', 'A-->QK_I']
df = pd.DataFrame([(l, h, round(score, 4), 
    round(mr(eval_head_lens)(_results, selected_model, l, h, strict=False).item(), 4), 
    round(mr(eval_head_lens)(_results, selected_model, l, h, strict=True).item(), 4), 
    round(ap_scores[attn_patterns[0]].mean().item(), 4),
    round(ap_scores[attn_patterns[1]].mean().item(), 4),
    round(ap_scores[attn_patterns[2]].mean().item(), 4),
    round(ap_scores[attn_patterns[3]].mean().item(), 4),
    round(ap_scores[attn_patterns[4]].mean().item(), 4),
    round(ap_scores[attn_patterns[5]].mean().item(), 4))
    for (l, h), score in d.top_heads.items() if (ap_scores := mr(get_head_matching_scores)(_results, attn_patterns, selected_model, l, h))],
    columns=['layer', 'head', 'score', 'acc', 'acc0'] + attn_patterns)
# 以 TSV 形式打印，便于后续复制或筛选
print(df.to_csv(sep='\t', index=True))
# _ = plt.figure(figsize=(8, 2)); _ = sns.heatmap(np.array(attr[60:, :, 0].cpu()), cbar=True)

In [ ]:
# 按阈值从 DataFrame 中挑选候选 attn_k 节点
nodes = [Node(int(l), int(h), 'attn_k', attn_pattern='A-->V') for l, h, s, acc, acc0, aps1, aps2 in df.values[:25]
         if int(h) < H and aps2 > 0.5]
# 打印候选节点，人工确认
for n in nodes: print(n)

In [ ]:
# 手动指定一个上游节点并连边到当前树节点
nodes = [Node(59, 21, 'attn_k', attn_pattern='A-->V')]
for n in nodes: add_edges(graph, n, tnode.data.nodes, tnode.data.attr)
# 基于新增节点继续向上扩展归因树
tnode = add_tnode(_results, selected_model, nodes, parent=tnode); print_tree(root)

In [ ]:
# 检查 attn_attrs_ds 的数据
for (l, h), aa in tnode.data.attn_attrs_ds.items():
    print(f"Head ({l}, {h}):")
    print(f"  Shape: {aa.shape}")
    print(f"  Min: {aa.min().item():.6f}, Max: {aa.max().item():.6f}")
    print(f"  Sum per row (first 3): {aa.sum(dim=-1)[:3].tolist()}")
    print(f"  Has NaN: {torch.isnan(aa).any().item()}")
    print(f"  Has Inf: {torch.isinf(aa).any().item()}")
    print(f"  All zeros rows: {(aa.sum(dim=-1) == 0).sum().item()}")
    print()

In [ ]:
d = tnode.data
d.groups, d.metrics, _, _ = cluster_heads(d.attn_attrs_ds, threshold=0.45, strengths=d.top_heads, model_config=selected_model.config,
    figsize=(12, 3), width_ratios=(4, 1), bar_height=0.5, leaf_font_size=9)

In [ ]:
d = tnode.data
d.groups, d.metrics, _, _ = cluster_heads(d.attn_attrs_ds, threshold=0.4, strengths=d.top_heads, model_config=selected_model.cfg,
    figsize=(12, 3), width_ratios=(4, 1), bar_height=0.5, leaf_font_size=9)

In [ ]:
attn_patterns = ['A-->V', 'A-->A-', 'V->VK_C', 'V->VK_I', 'A-->QK_C', 'A-->QK_I']
df = pd.DataFrame([(l, h, round(score, 4), 
    round(mr(eval_head_lens)(_results, selected_model, l, h, strict=False).item(), 4), 
    round(mr(eval_head_lens)(_results, selected_model, l, h, strict=True).item(), 4), 
    round(ap_scores[attn_patterns[0]].mean().item(), 4),
    round(ap_scores[attn_patterns[1]].mean().item(), 4),
    round(ap_scores[attn_patterns[2]].mean().item(), 4),
    round(ap_scores[attn_patterns[3]].mean().item(), 4),
    round(ap_scores[attn_patterns[4]].mean().item(), 4),
    round(ap_scores[attn_patterns[5]].mean().item(), 4))
    for (l, h), score in d.top_heads.items() if (ap_scores := mr(get_head_matching_scores)(_results, attn_patterns, selected_model, l, h))],
    columns=['layer', 'head', 'score', 'acc', 'acc0'] + attn_patterns)
print(df.to_csv(sep='\t', index=True))
# _ = plt.figure(figsize=(8, 2)); _ = sns.heatmap(np.array(attr[60:, :, 0].cpu()), cbar=True)

In [ ]:
for cluster_id, heads in d.groups.items():
    for l, h in heads:
        if cluster_id == 1: t, attn_pattern = 'attn_q', 'A-->V'
        elif cluster_id == 2: t, attn_pattern = 'attn_v', 'A-->A-'
        else: continue
        n = Node(l, h, t, attn_pattern=attn_pattern)
        add_edges(graph, n, d.nodes, d.attr)

In [ ]:
nodes = [Node(int(l), int(h), 'attn_q', attn_pattern='A-->V') for l, h, s, acc, acc0, aps1, aps2, aps3, aps4, aps5, aps6 in df.values[:25]
         if int(h) < H and aps1 > 0.3]
for n in nodes: print(n)

In [ ]:
nodes = [Node(int(l), int(h), 'attn_k', attn_pattern='A-->V') for l, h in [(55, 22)]]
for n in nodes: add_edges(graph, n, tnode.data.nodes, tnode.data.attr)
tnode = add_tnode(_results, selected_model, nodes, parent=tnode); print_tree(root)

In [ ]:
d = tnode.data
d.groups, d.metrics, _, _ = cluster_heads(d.attn_attrs_ds, threshold=0.4, strengths=d.top_heads, model_config=selected_model.cfg,  # thld 0.5->0.4
    figsize=(18, 5), width_ratios=(4, 1), bar_height=0.5, leaf_font_size=9)

In [ ]:
d = tnode.data
d.groups, d.metrics, _, _ = cluster_heads(d.attn_attrs_ds, threshold=0.4, strengths=d.top_heads, model_config=selected_model.config,  # thld 0.5->0.4
    figsize=(18, 5), width_ratios=(4, 1), bar_height=0.5, leaf_font_size=9)

In [ ]:
attn_patterns = ['A-->V', 'A-->A-', 'V->VK_C', 'V->VK_I', 'A-->QK_C', 'A-->QK_I']
df = pd.DataFrame([(l, h, round(score, 4), 
    round(mr(eval_head_lens)(_results, selected_model, l, h, strict=False).item(), 4), 
    round(mr(eval_head_lens)(_results, selected_model, l, h, strict=True).item(), 4), 
    round(ap_scores[attn_patterns[0]].mean().item(), 4),
    round(ap_scores[attn_patterns[1]].mean().item(), 4),
    round(ap_scores[attn_patterns[2]].mean().item(), 4),
    round(ap_scores[attn_patterns[3]].mean().item(), 4),
    round(ap_scores[attn_patterns[4]].mean().item(), 4),
    round(ap_scores[attn_patterns[5]].mean().item(), 4))
    for (l, h), score in d.top_heads.items() if (ap_scores := mr(get_head_matching_scores)(_results, attn_patterns, selected_model, l, h))],
    columns=['layer', 'head', 'score', 'acc', 'acc0'] + attn_patterns)
print(df.to_csv(sep='\t', index=True))
# _ = plt.figure(figsize=(8, 2)); _ = sns.heatmap(np.array(attr[60:, :, 0].cpu()), cbar=True)

In [ ]:
nodes = [Node(int(l), int(h), 'attn_q', attn_pattern='A-->V') for l, h, s, acc, acc0, aps1, aps2, aps3, aps4, aps5, aps6 in df.values[:25]
         if int(h) < H and aps1 > 0.5]
for n in nodes: print(n)

In [ ]:
for cluster_id, heads in d.groups.items():
    for l, h in heads:
        if cluster_id == 3: t, attn_pattern = 'attn_q', 'A-->V'
        else: continue
        n = Node(l, h, t, attn_pattern=attn_pattern)
        add_edges(graph, n, d.nodes, d.attr)

In [ ]:
nodes = [Node(46, 0, 'attn_k', attn_pattern='V->VK_C')]
for n in nodes: add_edges(graph, n, tnode.data.nodes, tnode.data.attr)
tnode = add_tnode(_results, selected_model, nodes, parent=tnode); print_tree(root)

In [ ]:
d = tnode.data
d.groups, d.metrics, _, _ = cluster_heads(d.attn_attrs_ds, threshold=0.5, strengths=d.top_heads, model_config=selected_model.config,
    figsize=(12, 3), width_ratios=(4, 1), bar_height=0.5, leaf_font_size=9)

In [ ]:
attn_patterns = ['A-->V', 'A-->A-', 'V->VK_C', 'V->VK_I', 'A-->QK_C', 'A-->QK_I']
df = pd.DataFrame([(l, h, round(score, 4), 
    round(mr(eval_head_lens)(_results, selected_model, l, h, strict=False).item(), 4), 
    round(mr(eval_head_lens)(_results, selected_model, l, h, strict=True).item(), 4), 
    round(ap_scores[attn_patterns[0]].mean().item(), 4),
    round(ap_scores[attn_patterns[1]].mean().item(), 4),
    round(ap_scores[attn_patterns[2]].mean().item(), 4),
    round(ap_scores[attn_patterns[3]].mean().item(), 4),
    round(ap_scores[attn_patterns[4]].mean().item(), 4),
    round(ap_scores[attn_patterns[5]].mean().item(), 4))
    for (l, h), score in d.top_heads.items() if (ap_scores := mr(get_head_matching_scores)(_results, attn_patterns, selected_model, l, h))],
    columns=['layer', 'head', 'score', 'acc', 'acc0'] + attn_patterns)
print(df.to_csv(sep='\t', index=True))
# _ = plt.figure(figsize=(8, 2)); _ = sns.heatmap(np.array(attr[60:, :, 0].cpu()), cbar=True)

In [ ]:
colored_tokens_multi(*show_attn(random.choice(_results), selected_model, 51, 10, downstreams=tnode.data.nodes, start=_results[0].index_map[0]['start']))#, start=100))

In [ ]:
for cluster_id, heads in d.groups.items():
    for l, h in heads:
        if cluster_id == 1: t, attn_pattern = 'attn_v', 'A-->A-'
        elif cluster_id == 2: t, attn_pattern = 'attn_q', 'A-->QK_C'
        elif cluster_id == 3: t, attn_pattern = 'attn_q', 'A-->V'
        elif cluster_id == 4: t, attn_pattern = 'attn_v', 'A-->X'
        elif cluster_id == 5: t, attn_pattern = 'attn_v', 'A-->QV'
        else: continue
        n = Node(l, h, t, attn_pattern=attn_pattern)
        add_edges(graph, n, d.nodes, d.attr)

In [ ]:
nodes = [Node(l, h, 'attn_v', attn_pattern='A-->V') for l, h in [(50, 60)]]
for n in nodes: add_edges(graph, n, tnode.data.nodes, tnode.data.attr)
tnode = add_tnode(_results, selected_model, nodes, parent=tnode); print_tree(root)

In [ ]:
tnode.data.top_heads

In [ ]:
d = tnode.data
d.groups, d.metrics, _, _ = cluster_heads(d.attn_attrs_ds, threshold=0.5, strengths=d.top_heads, model_config=selected_model.config,
    figsize=(12, 3), width_ratios=(4, 1), bar_height=0.5, leaf_font_size=9)

In [ ]:
attn_patterns = ['A-->V', 'A-->A-', 'V->VK_C', 'V->VK_I', 'A-->QK_C', 'A-->QK_I']
df = pd.DataFrame([(l, h, round(score, 4), 
    round(mr(eval_head_lens)(_results, selected_model, l, h, strict=False).item(), 4), 
    round(mr(eval_head_lens)(_results, selected_model, l, h, strict=True).item(), 4), 
    round(ap_scores[attn_patterns[0]].mean().item(), 4),
    round(ap_scores[attn_patterns[1]].mean().item(), 4),
    round(ap_scores[attn_patterns[2]].mean().item(), 4),
    round(ap_scores[attn_patterns[3]].mean().item(), 4),
    round(ap_scores[attn_patterns[4]].mean().item(), 4),
    round(ap_scores[attn_patterns[5]].mean().item(), 4))
    for (l, h), score in d.top_heads.items() if (ap_scores := mr(get_head_matching_scores)(_results, attn_patterns, selected_model, l, h))],
    columns=['layer', 'head', 'score', 'acc', 'acc0'] + attn_patterns)
print(df.to_csv(sep='\t', index=True))
# _ = plt.figure(figsize=(8, 2)); _ = sns.heatmap(np.array(attr[60:, :, 0].cpu()), cbar=True)

In [ ]:
colored_tokens_multi(*show_attn(random.choice(_results), selected_model, 30, 14, downstreams=tnode.data.nodes, start=_results[0].index_map[0]['start']))#, start=100))

In [ ]:
from vis import merge_gqa_groups, visualize_graph
for upstream, downstream in g.edges:
    if downstream == lmhead: g.edges[(upstream, downstream)] /= 10.
try:
    vg = merge_gqa_groups(g, selected_model.config)
    visualize_graph(vg, selected_model.config)
finally:
    for upstream, downstream in g.edges:
        if downstream == lmhead: g.edges[(upstream, downstream)] *= 10.

In [ ]:
candidates = join_lists([r.answers for r in results], dedup=True)  # ['K', 'B', 'G', 'R', 'Y']
logits_mask = (torch.ones(model.config.vocab_size) * (-1e4)).to(model.device, dtype=torch.float16)
logits_mask[tokenizer.encode(' ' + ' '.join(candidates))] = 0
mask = torch.eye(model.config.num_attention_heads).to(model.device, dtype=torch.float16) # n*n
pos_ids = torch.LongTensor(r.answer_indices) - 1

In [ ]:
heads = [
    # (77, 64), 
    # (76, 64), 
    # (77, 54), 
    # (79, 10), 
    # (78, 61), 
    # (76, 26), 
    # (72, 11), 
    # (73, 64), 
    # (76, 35),
    (73, 1),  # right_of
]
for r in _results:
    indices = torch.LongTensor(r.answer_indices) - 1
    sum_output = sum(get_head_output(selected_model, r.outputs, layer, head, indices) if head < N
        else r.outputs.mlp_outputs[layer][:, indices].to(selected_model.device) for layer, head in heads)
    norm = model.model.norm
    norm.variance = r.outputs.ln_states[-1][:, indices].to(model.device)
    sum_output = model.model.norm(sum_output)
    norm.variance = None
    # sum_output = r.outputs.hidden_states[-1][:, indices].to(model.device)
    top_pred = model.lm_head(sum_output).log_softmax(dim=-1).max(dim=-1)
    top_pred.values.mean()
    [t.replace('Ġ', ' ') for t in tokenizer.convert_ids_to_tokens(top_pred.indices[0])]
    r.responses
    [' ' + a for a in r.answers]

In [ ]:
with pd.option_context('display.max_colwidth', None, 'display.width', None):
    df.groupby(["model", "n_train", "rel_fn"])[['is_corrects', 'logprobs']].agg(mean_elementwise)

In [ ]:
with pd.option_context('display.max_colwidth', None, 'display.width', None):
    df_temp = df.copy(); df_temp['is_at'] = df_temp['rel_fn'] == 'at'
    df_temp.groupby(["model", "n_train", "is_at"]).agg(
        acc_elementwise=('is_corrects', mean_elementwise), acc=('is_corrects', mean_of_mean),
        logprobs_elementwise=('logprobs', mean_elementwise), logprobs=('logprobs', mean_of_mean))

In [ ]:
system_content = "You are a helpful assistant."
# system_content = "You are an expert at inferring patterns from grid puzzles and Python code examples."
self = client = LLMClient(provider=Provider.OPENROUTER, cache_dir=f"{os.getcwd()}/cache", system_content=None)
# self.api_key = "sk-bpacxulmtnysbweozezxsqvrhzckcobjtrvzomkcqlqdncek"
# self.client = OpenAI(api_key=self.api_key, base_url="https://api.siliconflow.cn/v1")

In [ ]:
results = client.generate_parallel(results=results, temperature=0.0, max_tokens=800, num_workers=8)
for r in results:
    r.model = r.model.value
    if len(r.response) >= 5: print(r.response)
    r.is_correct = r.response.strip('"').strip("'") == r.answer
df = pd.DataFrame([r.__dict__ for r in results])

In [ ]:
df.groupby(["model", "n_train", df["rel_pos"].apply(lambda t: sum(abs(x) for x in t))]).is_correct.mean()

In [ ]:
response = client.client.chat.completions.create(
    model=OpenRouterModels.QWEN3_32B.value,
    messages=[
        # {
        #     "role": "system",
        #     "content": self.system_content
        # },
        {
            "role": "user",
            "content": prompt0
        }
    ],
    temperature=0.0,
    max_tokens=4096,
    top_p=1.0,
    n=1
)

In [ ]:
p = json.load(open('../ARC-AGI/data/training/f8a8fe49.json'))

In [ ]:
# p = gen_puzzle(partial(gen_variation_color, special_var_idx=0), n_train=4)

def concat_side_by_side(g1, g2, sep=1, fill=Color.WHITE):
    a = np.array(g1)
    b = np.array(g2)
    h = max(a.shape[0], b.shape[0])
    def pad_to_h(x):
        pad_h = h - x.shape[0]
        if pad_h > 0:
            return np.vstack([x, np.full((pad_h, x.shape[1]), fill, dtype=x.dtype)])
        return x
    a_pad = pad_to_h(a)
    b_pad = pad_to_h(b)
    spacer = np.full((h, sep), fill, dtype=a_pad.dtype)
    return np.concatenate([a_pad, spacer, b_pad], axis=1)

# make figures 50% smaller
# plt.rcParams['figure.figsize'] = [plt.rcParams['figure.figsize'][0]*0.5, plt.rcParams['figure.figsize'][1]*0.5]
plt.rcParams['figure.figsize'] = [3.2, 2.4]

for pair in p['train'] + p['test']:
    combined = concat_side_by_side(pair['input'], pair['output'], sep=1, fill=Color.WHITE)
    show_colored_grid(np.array(combined).T, text=False)

In [ ]:
client = OpenAI(
    base_url="https://router.huggingface.co/v1",
    api_key=os.environ["HF_API_KEY"],
)
completion = client.chat.completions.create(
    model="barc0/Llama-3.1-ARC-Potpourri-Transduction-8B:featherless-ai",
    messages=[
        {
            "role": "system",
            "content": "You are a world-class puzzle solver with exceptional pattern recognition skills. "
                       "Your task is to analyze puzzles, spot patterns, and provide direct solutions."
        },
        {
            "role": "user",
            "content": prompt
        }
    ],
    temperature=0.3,
)
response = completion.choices[0].message.content
grid_text = re.findall(r"```(.*?)```", response, flags=re.S)[0].strip(); print(grid_text)

In [ ]:
show_colored_grid(np.array(text2grid(grid_text)).T, text=False)

In [ ]:
import json
import re
from pathlib import Path

seed_dir = Path('seeds')
id_pattern = re.compile(r'^[0-9a-f]{8}\.py$')
puzzle_ids = sorted({path.stem for path in seed_dir.glob('*.py') if id_pattern.match(path.name)})

In [ ]:
data_root = Path('../ARC-AGI/data')
candidate_subdirs = ['training', 'evaluation']
data_dirs = [data_root / sub for sub in candidate_subdirs if (data_root / sub).exists()]

total_elements_by_id = {}
n_train = []
for puzzle_id in puzzle_ids:
    json_path = None
    for directory in data_dirs:
        candidate = directory / f'{puzzle_id}.json'
        if candidate.exists():
            json_path = candidate
            break

    with json_path.open() as f:
        task = json.load(f)

    element_count = 0
    for section in ['train', 'test'][:1]:
        if section == 'train': n_train.append(len(task['train']))
        for pair in task.get(section, [])[:3]:
            for key in ('input', 'output'):
                grid = pair.get(key, [])
                element_count += sum(len(row) for row in grid)

    total_elements_by_id[puzzle_id] = element_count

print(f'Collected {len(puzzle_ids)} puzzle ids from {seed_dir}')
print(f'Computed element counts for {len(total_elements_by_id)} puzzles.')
# total_elements_by_id

In [ ]:
data_root = Path('/home/xd/projects/ARC-AGI/data')
all_tasks = {}
for split in ('training', 'evaluation'):
    split_dir = data_root / split
    for json_path in split_dir.glob('*.json'):
        puzzle_id = json_path.stem
        with json_path.open() as f: all_tasks[puzzle_id] = json.load(f)

In [ ]:
for task in all_tasks.values():
    for section in ("train", "test"):
        for pair in task.get(section, []):
            annotate_pair(pair)

In [ ]:
puzzle_id = "007bbfb7"
task = all_tasks[puzzle_id]
infer_test_output_shape(task, (3, 3))
# all(pair['input_num_subgrids'] != (1, 1) for pair in task['train'])
# not (any(pair['input_num_subgrids'][0] == 1 for pair in task['train']) and any(pair['input_num_subgrids'][1] == 1 for pair in task['train']))
# not any(pair['input_sep_color_pct'] > 0.55 and not (min(pair['input_num_subgrids']) >= 3 and pair['input_num_subgrids'] == pair['output_num_subgrids']) for pair in task['train'])

In [ ]:
tasks = {}
for puzzle_id, task in all_tasks.items():
    if (
        not all(np.array(pair['input']).shape == np.array(pair['output']).shape for pair in task['train']) and
        not len(set(tuple(ns[1]/ns[0] for ns in zip(np.array(pair['input']).shape, np.array(pair['output']).shape)) for pair in task['train'])) == 1 and
        # len(set(tuple(math.log(ns[1], max(ns[0], 1.1)) for ns in zip(np.array(pair['input']).shape, np.array(pair['output']).shape)) for pair in task['train'])) == 1 and
        not len(set(np.array(pair['output']).shape for pair in task['train'])) == 1 and 
        has_subgrids(task) and
        not all(pair['input_num_subgrids'] == np.array(pair['output']).shape for pair in task['train']) #and # 3
        not all(all(np.array(pair['output']).shape == shape for shape in pair['input_subgrid_shapes']) for pair in task['train']) # 2
    ):
        tasks[puzzle_id] = task
len(tasks)

In [ ]:
# f76d97a5/27a28665 1x1 content blocks
# ba97ae07 empty content blocks
# 90f3ed37 all 1x1 except one nx1/1xn
# ecdecbb3 mx1, 1xn, px1
# 543a7ed5/b2862040/ce602527/2c608aff non-black background!!
# de1cd16c/e6721834 use color instead of sep to divide content blocks!!
# 995c5fa3 sep's area larger than content blocks!!
# 5daaa586/0520fde7 !!
# c909285e symmetric texture
# 662c240a divide content blocks w/o sep!!
# e48d4e1a perfect but not sep!!!!

# 29ec7d0e sep is cropped for completion!
# 4be741c5 no sep!
# 85b81ff1 black sep
# e6721834 (3, 1) (1, 3) (1, 4) troublesome!5
77fdfe62
{'input_num_subgrids': (3, 3), 'input_sep_color_pct': 0.4375, 'output_num_subgrids': (1, 1)}
{'input_num_subgrids': (3, 3), 'input_sep_color_pct': 0.5555555555555556, 'output_num_subgrids': (1, 1)}
{'input_num_subgrids': (3, 3), 'input_sep_color_pct': 0.4375, 'output_num_subgrids': (1, 1)}